In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import

In [2]:
!pip install sentence_transformers
from sentence_transformers import SentenceTransformer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 71.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 36.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 83.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.7 MB/s eta 0:00:00
  Created wheel for sentence_transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=0e71ca4d8e881384f6a52beaf2f7e75bff814df4cb6d695223d1ad805fde0d15
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence_transformers


In [3]:
import re
import pandas as pd
import numpy as np
import random
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

## Random Seed

In [4]:
SEED = 0

np.random.seed(SEED)
random.seed(SEED)

## Load Data

In [5]:
df = pd.read_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/news.csv')
# df.head()

In [6]:
# 제목 + 내용
df['text'] = df['title'] + '. ' + df['contents']
# df.head()

In [7]:
# from collections import Counter

# # 데이터프레임 내의 텍스트를 모두 하나의 문자열로 합치기
# all_text = ' '.join(df['text'])

# # 모든 단어 추출 (단어는 공백을 기준으로 분리)
# words = all_text.split()

# # 단어 빈도 계산
# word_counts = Counter(words)

# # 가장 빈도가 높은 단어 출력 (예: 상위 10개)
# most_common_words = word_counts.most_common(50)

# # 결과 출력
# for word, count in most_common_words:
#     print(f'{word}: {count}번 반복됨')

# stopwords = ["the", "and", "in", "is", "at", "on", "it", "with", "an", "as", "of"]
# stopwords = ['the:', 'to', 'of', 'a', 'in', 'and', 'on', 'for', 'that', 'is', 'with', 'as', 'at',
#              'has', 'by', 'its', 'it', 'from', 'was', 'be', 'will', 'an', 'his', 'have', 'are',
#              'after', 'their', 'he', 'over', 'not', 'this', 'but', 'about', 'more']

## Pre-processing

In [8]:
# ["the", "and", "in", "is", "at", "on", "it", "with", "an", "as", "of"]

In [9]:
# stopwords
stopwords = ['the', 'to', 'of', 'a', 'in', 'and', 'on', 'for', 'that', 'is', 'with', 'as', 'at',
             'has', 'by', 'its', 'it', 'from', 'was', 'be', 'will', 'an', 'his', 'have', 'are',
             'after', 'their', 'he', 'over', 'not', 'this', 'but', 'about', 'more', 'her']

def preprocess_text(text):
    # 텍스트에서 영어 불용어 제거
    text = ' '.join([word for word in text.split() if word.lower() not in stopwords])

    # 정규 표현식을 사용하여 괄호와 함께 나오는 "AP"와 "Reuters"를 제거합니다.
    text = re.sub(r'\(AP\)|\(Reuters\)|AP', '', text)

    # 문자열 ";p", "/p"를 모두 삭제
    text = re.sub(r'[;/]p', '', text)

    # HREF로 시작하는 패턴 모두 삭제
    text = re.sub(r'HREF="[^"]*"', '', text)

    # '/'와 '\' 문자 모두 삭제
    text = re.sub(r'[\/\\]', '', text)

    # 해시태그 제거
    text = re.sub(r'#\w+', '', text)

    text = re.sub(r'[#;\$#]', '', text)

    # 공백 및 특수문자 제거
    text = re.sub(r'\s+', ' ', text).strip()

    # 정규 표현식을 사용하여 HTML 엔터티 코드를 제거합니다.
    text = re.sub(r'&#[0-9]+;', '', text)

    # 숫자 제거
    text = re.sub(r'\d+', '', text)

    return text

In [10]:
df['processed_text'] = df['text'].apply(preprocess_text)

In [11]:
# from collections import Counter

# # 데이터프레임 내의 텍스트를 모두 하나의 문자열로 합치기
# all_text = ' '.join(df['processed_text'])

# # 모든 단어 추출 (단어는 공백을 기준으로 분리)
# words = all_text.split()

# # 단어 빈도 계산
# word_counts = Counter(words)

# # 가장 빈도가 높은 단어 출력 (예: 상위 100개)
# most_common_words = word_counts.most_common(100)

# # 결과 출력
# for word, count in most_common_words:
#     print(f'{word}: {count}번 반복됨')

## Feature Extraction

In [12]:
# Sentence BERT 모델 로드
model = SentenceTransformer('all-mpnet-base-v2')

# 텍스트 feature 추출
sentence_embeddings = model.encode(df['processed_text'].tolist(), show_progress_bar=True)

from sklearn.preprocessing import MinMaxScaler

# 임베딩을 표준화
scaler = MinMaxScaler()
scaled_embeddings = scaler.fit_transform(sentence_embeddings)

# 추출한 feature를 데이터프레임에 저장
df_embeddings = pd.DataFrame(scaled_embeddings)

Batches:   0%|          | 0/1875 [00:00<?, ?it/s]

## Clustering

In [13]:
# Sentence BERT 임베딩을 사용하여 군집화 수행
kmeans = KMeans(n_clusters=6, random_state=SEED)

df['kmeans_cluster'] = kmeans.fit_predict(scaled_embeddings)

## Post-processing

### World: 0 -> 5

In [14]:
df[df['kmeans_cluster'] == 0]['text']

1        Bruce Lee statue for divided city. In Bosnia, ...
29       Israel Kills 3 Palestinians in Big Gaza Incurs...
34       The Folly of the Sole Superpower Writ Small au...
56       Sadr #39;s aide denies entering of Iraqi polic...
57       Former Nazi Guard Loses Canadian Court Ruling ...
                               ...                        
59982    Militants kill 12 in J amp;K ahead of PMs visi...
59984    The Plot Thickens: Testing European Tolerance....
59987    Nepal Seeks to Break Rebel Siege with Air Patr...
59992    US troops on offensive in Iraq. US troops went...
59999    Cassini Craft Spies Saturn Moon Dione (AP). AP...
Name: text, Length: 9642, dtype: object

### Business: 1 -> 0

In [15]:
df[df['kmeans_cluster'] == 1]['text']

20       Deere's Color Is Green. With big tractors, big...
27       Kmart-Sears merger about price, quality. Avera...
36       Agencies Postpone Issuing New Rules Until Afte...
37       Deep Impact Space Probe Aims to Slam Into Come...
40       Out for V-I-C-T-O-R-Y, but Missing Tiles. Miss...
                               ...                        
59985    European Shares Tread Water (Reuters). Reuters...
59986    Higher trade growth predicted in 2004 despite ...
59994    Chain Store Sales Rise in the Latest Week. NEW...
59996    After Steep Drop, Price of Oil Rises. The free...
59998    Albertsons on the Rebound. The No. 2 grocer re...
Name: text, Length: 12052, dtype: object

### Politics: 2 -> 2

In [16]:
df[df['kmeans_cluster'] == 2]['text']

7        Bump Stock Maker Resumes Sales One Month After...
8        Obama Marks Anniversary Of 9/11 Attacks With M...
9        Republican Congressman Says Trump Should Apolo...
11       Kerry rolls out tax-cut plan for middle class....
12       Read Live Updates From The South Carolina Demo...
                               ...                        
59953    A Dozen Lessons Donald Trump Could Learn From ...
59954    You Don't Have To Agree With Donald Trump To B...
59959    US Era of Dominance Is Dwindling as China Take...
59966    Mitt Romney Lambasts Donald Trump As A 'Phony'...
59988    Obama To Call For More Icebreakers In The Arct...
Name: text, Length: 10637, dtype: object

### Tech: 3 -> 4

In [17]:
df[df['kmeans_cluster'] == 3]['text']

3        Macromedia contributes to eBay Stores. Macrome...
4        Qualcomm plans to phone it in on cellular repa...
5        Thomson to Back Both Blu-ray and HD-DVD. Compa...
23       FTC Files First Lawsuit Against Spyware Concer...
31       Sony PSP Draws Crowds and Lines on First Day (...
                               ...                        
59972    SBC Details Fiber Plans. SBC Communications In...
59975    Supreme Court Won #39;t Weigh Net Music Lawsui...
59976    The Sims Go to College. November 22, 2004 - It...
59978    SMIC to challenge latest TSMC infringement cla...
59983    Partners Weigh In On Firefox, IE Faceoff. Solu...
Name: text, Length: 9354, dtype: object

### Sports: 4 -> 3

In [18]:
df[df['kmeans_cluster'] == 4]['text']

0        Spanish coach facing action in race row. MADRI...
6        Time to Talk Baseball. It's time to talk about...
13       GAME DAY PREVIEW Game time: 6:00 PM. CHARLOTTE...
16       Fischer's Fiancee: Marriage Plans Genuine (AP)...
22       College Basketball: Georgia Tech, UConn Win. A...
                               ...                        
59990    Wizards Down Galaxy. Kansas City secures the t...
59991    Irish talk of softening schedule a little. whi...
59993    Olson: Hoping to preserve the joy of Sox. Sinc...
59995    Dolphins Break Through, Rip Rams For First Win...
59997    Pro football: Culpepper puts on a show. To say...
Name: text, Length: 11588, dtype: object

### Entertainment: 5 -> 1

In [19]:
df[df['kmeans_cluster'] == 5]['text']

2        Only Lovers Left Alive's Tilda Swinton Talks A...
10       Harry #39;s argy-bargy. PRINCE Charles has ask...
21       Blake Leeper Wants to Be the First American Pa...
28       Cate Blanchett Set To Star As Lucille Ball In ...
45       The Trouble with Broadcasting in a Social Worl...
                               ...                        
59941    7 Honest Mistakes That Can Get You Fired autho...
59964    Russell Crowe Reaps Shocking Sum In Divorce Au...
59967    Europeans in Thrall of American Culture (AP). ...
59969    'American Hustle' '12 Years a Slave' Lead 2014...
59981    Khloe Kardashian Gets A Hilarious Lesson In Po...
Name: text, Length: 6727, dtype: object

### Mapping

In [20]:
mapping_dict = {
    0: 5,
    1: 0,
    2: 2,
    3: 4,
    4: 3,
    5: 1
}

In [21]:
df['mapping'] = df['kmeans_cluster'].apply(lambda x: mapping_dict[x])

## Submission

In [22]:
sample = pd.read_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/sample_submission.csv')

In [23]:
sample['category'] = df['mapping'].values
sample['category'].head()

0    3
1    5
2    1
3    4
4    4
Name: category, dtype: int64

In [24]:
sample.to_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/all-mpnet-base-v2_25.csv', index=False)